# 🤖 MINT in Action – Bau deinen eigenen Dokumenten-Assistenten!

## Was machen wir heute?
Wir bauen Schritt für Schritt einen **RAG-Assistenten** (Retrieval-Augmented Generation).
Das ist ein System, das:
1. 📄 Ein Dokument lesen kann
2. 🔍 Die richtige Stelle darin findet
3. 💬 Eine Antwort auf eure Frage formuliert

**Genau so funktioniert auch der KI-Assistent in unserem Dokumentenmanagementsystem!**

---

### 📋 Anleitung
- Führt die Zellen **der Reihe nach** aus (▶️ Button links oder `Shift+Enter`)
- Lest die Erklärungen in den Textblöcken
- Überall wo ihr 🔧 seht, könnt ihr Werte ändern und experimentieren
- Arbeitet in euren Gruppen zusammen – einer tippt, die anderen helfen!

### 💻 Dieses Notebook läuft im Browser – kein Setup nötig!
Alle Pakete sind vorinstalliert. Einfach loslegen!

In [ ]:
# ============================================================
# 🔧 SETUP – Modus wählen
# ============================================================
# Auf Binder sind alle Pakete schon vorinstalliert!
# Falls ihr lokal arbeitet und Pakete fehlen, entkommentiert die nächste Zeile:
# %pip install pymupdf faiss-cpu sentence-transformers matplotlib scikit-learn transformers accelerate torch openai

# ⚡ MODUS AUSWÄHLEN:
# "OPEN_SOURCE" = Kostenlose Open-Source-Modelle (läuft ohne API-Key)
# "API"         = OpenAI API (braucht API-Key)

MODUS = "OPEN_SOURCE"  # 🔧 ÄNDERE MICH bei Bedarf!

# Nur bei API-Modus nötig:
OPENAI_API_KEY = ""

import warnings
warnings.filterwarnings('ignore')

print(f"✅ Modus: {MODUS}")
if MODUS == "OPEN_SOURCE":
    import torch
    if torch.cuda.is_available():
        print(f"🚀 GPU erkannt: {torch.cuda.get_device_name(0)}")
    else:
        print("💡 CPU-Modus – funktioniert, Antworten dauern ein paar Sekunden länger")

In [ ]:
# ============================================================
# 🧠 MODELLE LADEN
# ============================================================
# Hier laden wir die KI-Modelle, die wir gleich verwenden.
# Das kann 1-2 Minuten dauern – perfekte Zeit für Fragen!

import numpy as np

if MODUS == "OPEN_SOURCE":
    print("⏳ Lade Embedding-Modell (sentence-transformers)...")
    from sentence_transformers import SentenceTransformer
    embedding_modell = SentenceTransformer('all-MiniLM-L6-v2')
    print("✅ Embedding-Modell geladen!")

    print("\n⏳ Lade Sprachmodell (Flan-T5)...")
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
    llm_modell = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

    import torch
    if torch.cuda.is_available():
        llm_modell = llm_modell.to("cuda")
        print(f"✅ Sprachmodell geladen! (GPU: {torch.cuda.get_device_name(0)})")
    else:
        print("✅ Sprachmodell geladen! (CPU-Modus – etwas langsamer)")

elif MODUS == "API":
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("✅ OpenAI-Client bereit!")

print("\n🎉 Alle Modelle sind bereit! Weiter geht's!")

---
## 📄 Schritt 1: Text aus einem Dokument lesen

**Was passiert hier?**
Bevor die KI ein Dokument "verstehen" kann, muss sie es erstmal **lesen** können.
Für uns Menschen ist das trivial – wir schauen auf eine Seite und sehen Text.
Für den Computer ist ein PDF erstmal nur eine Sammlung von Pixeln oder Vektordaten.

**In der echten Welt:** Bei uns in der Firma macht das die **OCR** (Optical Character Recognition) –
die erkennt sogar Text in gescannten Papierdokumenten und Fotos!

**Jetzt:** Wir extrahieren den Text aus einem PDF. Gebt den Pfad zu einem PDF an
oder verwendet den vorbereiteten Beispieltext.

In [ ]:
# ============================================================
# 📄 SCHRITT 1: TEXT AUS EINEM DOKUMENT LESEN
# ============================================================

import fitz  # PyMuPDF
import os

# --- Option 1: Eigenes PDF laden (Pfad angeben) ---
# 🔧 Tragt hier den Pfad zu eurem PDF ein, oder lasst es leer für den Beispieltext:
pdf_pfad = ""  # z.B. "dokument.pdf" oder r"C:\Users\...\mein_dokument.pdf"

dateiname = None
if pdf_pfad and os.path.isfile(pdf_pfad):
    dateiname = pdf_pfad
    print(f"✅ Datei '{dateiname}' gefunden!")
else:
    if pdf_pfad:
        print(f"⚠️ Datei '{pdf_pfad}' nicht gefunden – verwende Beispieltext.")
    else:
        print("ℹ️ Kein PDF-Pfad angegeben – verwende Beispieltext.")

# --- Text extrahieren ---
if dateiname and dateiname.lower().endswith('.pdf'):
    dokument = fitz.open(dateiname)
    text = ""
    for seite in dokument:
        text += seite.get_text()
    anzahl_seiten = dokument.page_count
    dokument.close()
    print(f"\n📊 Statistik:")
    print(f"   Seiten: {anzahl_seiten}")
    print(f"   Zeichen: {len(text):,}")
    print(f"   Wörter: {len(text.split()):,}")
else:
    # Beispieltext über Machine Learning und Dokumentenverarbeitung
    text = (
        "Machine Learning ist ein Teilgebiet der Künstlichen Intelligenz. "
        "Es ermöglicht Computern, aus Daten zu lernen, ohne explizit programmiert zu werden. "
        "Der Begriff wurde 1959 von Arthur Samuel geprägt.\n\n"
        "Es gibt drei Hauptarten des maschinellen Lernens:\n\n"
        "Überwachtes Lernen (Supervised Learning): Der Algorithmus lernt aus gelabelten Daten. "
        "Beispiele sind Bilderkennung, Spamerkennung und Vorhersage von Hauspreisen. "
        "Das Modell bekommt Eingabe-Ausgabe-Paare und lernt die Zuordnung.\n\n"
        "Unüberwachtes Lernen (Unsupervised Learning): Der Algorithmus findet Muster in "
        "ungelabelten Daten. Beispiele sind Kundensegmentierung, Anomalieerkennung und "
        "Dimensionsreduktion. Das Modell sucht selbstständig nach Strukturen in den Daten.\n\n"
        "Bestärkendes Lernen (Reinforcement Learning): Ein Agent lernt durch Interaktion "
        "mit einer Umgebung. Er bekommt Belohnungen oder Strafen für seine Aktionen. "
        "Beispiele sind Spielstrategien, Robotersteuerung und autonomes Fahren.\n\n"
        "Neuronale Netze sind von biologischen Nervensystemen inspiriert. Sie bestehen aus "
        "Schichten von künstlichen Neuronen. Deep Learning verwendet tiefe neuronale Netze "
        "mit vielen Schichten. Dadurch können komplexe Muster erkannt werden.\n\n"
        "Transformer-Modelle haben 2017 die Verarbeitung natürlicher Sprache revolutioniert. "
        "Sie verwenden einen Aufmerksamkeitsmechanismus (Attention), der es erlaubt, "
        "Beziehungen zwischen allen Wörtern eines Textes gleichzeitig zu erfassen. "
        "GPT, BERT und T5 sind bekannte Transformer-Modelle.\n\n"
        "RAG (Retrieval-Augmented Generation) kombiniert Informationssuche mit Textgenerierung. "
        "Dabei wird zuerst in einer Wissensbasis nach relevanten Informationen gesucht. "
        "Diese werden dann als Kontext an ein Sprachmodell übergeben, das eine Antwort generiert. "
        "So kann das Modell auf aktuelle und spezifische Informationen zugreifen, "
        "die nicht in seinen Trainingsdaten enthalten sind.\n\n"
        "Vektordatenbanken speichern Texte als numerische Vektoren, sogenannte Embeddings. "
        "Ähnliche Texte haben ähnliche Vektoren. Dadurch kann man semantisch ähnliche "
        "Dokumente finden, auch wenn sie unterschiedliche Wörter verwenden.\n\n"
        "OCR (Optical Character Recognition) wandelt Bilder von Text in maschinenlesbaren "
        "Text um. Moderne OCR-Systeme verwenden Deep Learning und erreichen sehr hohe "
        "Genauigkeit. Sie können auch handgeschriebenen Text erkennen.\n\n"
        "Information Extraction identifiziert und extrahiert strukturierte Informationen "
        "aus unstrukturiertem Text. Beispiele sind die Erkennung von Namen, Daten, Beträgen "
        "und Adressen in Dokumenten wie Rechnungen oder Verträgen.\n\n"
        "Embeddings werden berechnet, indem Text durch ein neuronales Netz geschickt wird. "
        "Das Netz wurde auf riesigen Textmengen trainiert und hat gelernt, die Bedeutung "
        "von Wörtern und Sätzen in Zahlenvektoren zu kodieren. Ähnliche Bedeutungen "
        "führen zu ähnlichen Vektoren im hochdimensionalen Raum."
    )
    print("📝 Beispieltext geladen!")
    print(f"   Zeichen: {len(text):,}")
    print(f"   Wörter: {len(text.split()):,}")

# Textvorschau anzeigen
print(f"\n--- Vorschau (erste 500 Zeichen) ---")
print(text[:500])
print("...")

---
## ✂️ Schritt 2: Text in Stücke (Chunks) aufteilen

**Was passiert hier?**
Stellt euch vor, jemand fragt euch: "Was steht in dem Buch über Napoleon?"
Ihr lest nicht das ganze Buch nochmal durch – ihr schlagt das richtige **Kapitel** auf!

Genau das machen wir hier: Wir teilen den Text in kleine Stücke (**Chunks**) auf.
Später suchen wir dann nur die passenden Chunks zu einer Frage.

**Warum nicht den ganzen Text nehmen?**
- Sprachmodelle haben ein **Limit** (wie viel sie gleichzeitig lesen können)
- Kleinere Stücke = **präzisere** Suche
- Aber: Zu kleine Stücke = wichtiger Kontext geht verloren! ⚖️

**🔧 Experimentiert:** Ändert `chunk_groesse` und `ueberlappung` und seht was passiert!

In [ ]:
# ============================================================
# ✂️ SCHRITT 2: TEXT IN STÜCKE (CHUNKS) AUFTEILEN
# ============================================================

# --- Parameter zum Experimentieren ---
chunk_groesse = 300    # 🔧 Zeichen pro Chunk (probiert: 100, 300, 500, 1000)
ueberlappung = 50      # 🔧 Überlappung zwischen Chunks (probiert: 0, 50, 100)

def text_in_chunks_aufteilen(text, chunk_groesse=300, ueberlappung=50):
    """Teilt einen Text in überlappende Stücke (Chunks) auf."""
    chunks = []
    start = 0
    while start < len(text):
        ende = start + chunk_groesse
        chunk = text[start:ende].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_groesse - ueberlappung
    return chunks

# Text aufteilen
chunks = text_in_chunks_aufteilen(text, chunk_groesse, ueberlappung)

# Ergebnis anzeigen
print(f"📊 Ergebnis:")
print(f"   Originaltext: {len(text):,} Zeichen")
print(f"   Chunk-Größe: {chunk_groesse} Zeichen")
print(f"   Überlappung: {ueberlappung} Zeichen")
print(f"   Anzahl Chunks: {len(chunks)}")

print(f"\n--- Die ersten 3 Chunks ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'='*50}")
    print(f"📦 Chunk {i+1} ({len(chunk)} Zeichen):")
    print(f"{'='*50}")
    print(chunk)

---
## 🧮 Schritt 3: Embeddings – Text wird zu Zahlen

**Was passiert hier?**
Der Computer kann keine Wörter vergleichen – er braucht **Zahlen**.
Wir verwandeln jeden Chunk in einen **Vektor** (eine Liste von Zahlen).

**💡 Analogie: GPS-Koordinaten für Wörter**
- Jedes Wort bekommt "Koordinaten" in einem Zahlenraum
- **Ähnliche** Wörter liegen **nah** beieinander: "Hund" 📍 ↔ "Katze" 📍 (nah!)
- **Verschiedene** Wörter liegen **weit** weg: "Hund" 📍 ↔ "Mathematik" 📍 (weit!)

Nur dass unsere "Koordinaten" nicht 2 Dimensionen haben (wie echtes GPS),
sondern **384 Dimensionen**! Das können wir uns nicht vorstellen – deshalb
reduzieren wir es für die Visualisierung auf 2D.

**🤓 Für Fortgeschrittene:**
Das Embedding-Modell ist ein neuronales Netz (Transformer), das mit Millionen
von Texten trainiert wurde. Es hat gelernt, **Bedeutung** in Zahlen auszudrücken.

In [ ]:
# ============================================================
# 🧮 SCHRITT 3: EMBEDDINGS ERZEUGEN
# ============================================================

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 10

# --- Embeddings berechnen ---
print("⏳ Berechne Embeddings für alle Chunks...")

if MODUS == "OPEN_SOURCE":
    embeddings = embedding_modell.encode(chunks, show_progress_bar=True)
elif MODUS == "API":
    response = client.embeddings.create(
        input=chunks,
        model="text-embedding-3-small"
    )
    embeddings = np.array([item.embedding for item in response.data])

embeddings = np.array(embeddings)
print(f"\n✅ Embeddings berechnet!")
print(f"   Anzahl Chunks: {len(embeddings)}")
print(f"   Dimensionen pro Embedding: {embeddings.shape[1]}")
print(f"   → Jeder Chunk ist jetzt ein Vektor mit {embeddings.shape[1]} Zahlen!")

# --- Visualisierung: 2D-Landkarte ---
print("\n📊 Erstelle 2D-Visualisierung...")
n_components = min(2, len(chunks))
pca = PCA(n_components=n_components)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=range(len(chunks)), cmap='viridis',
    s=120, alpha=0.7, edgecolors='black', linewidth=0.5
)

for i in range(len(chunks)):
    plt.annotate(f'Chunk {i+1}',
        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
        fontsize=8, ha='center', va='bottom')

plt.colorbar(scatter, label='Position im Dokument')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('🗺️ Landkarte der Textabschnitte\n(Ähnliche Chunks liegen nah beieinander!)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Chunks die nah beieinander liegen, handeln von ähnlichen Themen!")
print("   Der Computer hat das SELBST herausgefunden – ohne Regeln von uns!")

---
## 🔍 Schritt 4: Ähnlichkeitssuche – Die richtige Stelle finden

**Was passiert hier?**
Jetzt können wir endlich **Fragen stellen**! Der Computer:
1. Wandelt eure Frage in einen Embedding-Vektor um
2. Sucht die Chunks, deren Vektoren am **nächsten** an der Frage liegen
3. Das sind die relevantesten Textabschnitte!

**💡 Analogie:** Das ist wie Google – aber für **EUER** Dokument.
Und statt Stichwörter zu matchen, versteht es die **Bedeutung**!

Ihr könnt z.B. fragen "Wie lernt eine Maschine?" und es findet Chunks
über "Machine Learning" – obwohl das Wort gar nicht in der Frage vorkommt!

**🔧 Experimentiert:** Ändert die `frage`-Variable und schaut was gefunden wird!

In [ ]:
# ============================================================
# 🔍 SCHRITT 4: ÄHNLICHKEITSSUCHE
# ============================================================

import faiss

# --- FAISS-Index aufbauen (unsere "Suchmaschine") ---
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print(f"✅ Suchindex erstellt mit {index.ntotal} Einträgen!")

# --- Frage stellen ---
frage = "Was ist Retrieval-Augmented Generation?"  # 🔧 ÄNDERE MICH!

print(f"\n❓ Frage: \"{frage}\"")
print(f"{'='*60}")

# Frage in Embedding umwandeln
if MODUS == "OPEN_SOURCE":
    frage_embedding = embedding_modell.encode([frage])
elif MODUS == "API":
    response = client.embeddings.create(input=[frage], model="text-embedding-3-small")
    frage_embedding = np.array([response.data[0].embedding])

# Die 3 ähnlichsten Chunks finden
k = 3  # 🔧 Anzahl Ergebnisse (probiert: 1, 3, 5)
abstande, indizes = index.search(frage_embedding.astype('float32'), k)

# --- Ergebnisse anzeigen ---
print(f"\n🏆 Top {k} relevanteste Textabschnitte:\n")
farben = ['🟢', '🟡', '🟠']
for rang, (idx, abstand) in enumerate(zip(indizes[0], abstande[0])):
    relevanz = max(0, 100 - abstand * 5)
    print(f"{farben[rang]} Platz {rang+1} | Chunk {idx+1} | Ähnlichkeit: {relevanz:.0f}%")
    print(f"   \"{chunks[idx][:150]}...\"")
    print()

# --- Visualisierung: Frage + Chunks auf der Karte ---
frage_2d = pca.transform(frage_embedding)

plt.figure(figsize=(12, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
            c='lightgray', s=80, alpha=0.5, edgecolors='gray', linewidth=0.5)

farb_werte = ['#2ecc71', '#f1c40f', '#e67e22']
for rang, idx in enumerate(indizes[0]):
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
               c=farb_werte[rang], s=200, edgecolors='black', linewidth=2,
               label=f'Platz {rang+1}: Chunk {idx+1}', zorder=5)

plt.scatter(frage_2d[0, 0], frage_2d[0, 1],
            c='red', marker='*', s=400, edgecolors='black',
            linewidth=2, label='⭐ Deine Frage', zorder=10)

for i in range(len(chunks)):
    plt.annotate(f'{i+1}', (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=7, ha='center', va='center')

plt.legend(loc='best', fontsize=10)
plt.title(f'🔍 Suche: \"{frage}\"\n(Stern = Frage, farbige Punkte = Treffer)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Je näher ein Chunk am Stern liegt, desto relevanter ist er für die Frage!")

---
## 💬 Schritt 5: Antwort generieren – Das "G" in RAG!

**Was passiert hier?**
Jetzt kommt der letzte und coolste Schritt! Wir geben einem Sprachmodell:
- Den **Kontext** (die gefundenen Chunks)
- Die **Frage**

Und es formuliert eine menschlich klingende **Antwort** – basierend NUR auf dem Kontext!

**💡 Warum "nur basierend auf dem Kontext"?**
Das ist der entscheidende Trick: Wir sagen dem Modell explizit,
dass es **nichts erfinden** soll. Es soll nur Informationen aus den
gegebenen Textabschnitten verwenden. So vermeiden wir **Halluzinationen**
(erfundene Fakten).

**In der echten Welt:** Bei einem Dokumenten-Assistenten ist das SUPER wichtig!
Wenn ein Kunde nach einer Rechnungsnummer fragt, darf die KI keine Nummer erfinden!

In [ ]:
# ============================================================
# 💬 SCHRITT 5: ANTWORT GENERIEREN
# ============================================================

# Kontext aus den gefundenen Chunks
kontext = "\n\n".join([chunks[idx] for idx in indizes[0]])

# Prompt konstruieren
prompt = (
    "Beantworte die folgende Frage basierend NUR auf dem gegebenen Kontext.\n"
    'Wenn der Kontext die Antwort nicht enthält, sage '
    '"Das kann ich aus dem Dokument nicht beantworten."\n\n'
    f"Kontext:\n{kontext}\n\n"
    f"Frage: {frage}\n\n"
    "Antwort:"
)

print("📋 So sieht der Prompt aus, den das Modell bekommt:")
print(f"{'='*60}")
print(prompt[:600])
if len(prompt) > 600:
    print("... (gekürzt)")
print(f"{'='*60}")

# --- Antwort generieren ---
print("\n⏳ Generiere Antwort...")

if MODUS == "OPEN_SOURCE":
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        outputs = llm_modell.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)

elif MODUS == "API":
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Du bist ein hilfreicher Assistent, der Fragen basierend auf dem gegebenen Kontext beantwortet. Antworte auf Deutsch."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=300,
        temperature=0.7
    )
    antwort = response.choices[0].message.content

print(f"\n🤖 Antwort des Modells:")
print(f"{'='*60}")
print(f"\n{antwort}\n")
print(f"{'='*60}")
print(f"\n📚 Diese Antwort basiert auf {len(indizes[0])} Textabschnitten aus dem Dokument.")
print("💡 Das Modell hat NICHT sein eigenes Wissen benutzt, sondern NUR den Kontext!")

---
## 🚀 Schritt 6: Alles zusammen – Eure eigene RAG-Pipeline!

**Geschafft! 🎉**

Ihr habt gerade eine vollständige **Retrieval-Augmented Generation (RAG)**
Pipeline gebaut! Genau so funktioniert auch der KI-Assistent in unserem
Dokumentenmanagementsystem – nur mit mehr Daten, besseren Modellen
und einer schönen Benutzeroberfläche.

Die Funktion `frag_das_dokument()` fasst alles zusammen.
Stellt beliebige Fragen an euer Dokument!

### 🎮 Experimentier-Ideen:
1. Stellt mindestens 3 verschiedene Fragen
2. Stellt eine Frage, die das Dokument NICHT beantworten kann – was passiert?
3. Ändert `anzahl_chunks` – was passiert mit mehr oder weniger Kontext?
4. Ändert `pdf_pfad` in Schritt 1 auf ein anderes PDF und führt alles nochmal aus

In [ ]:
# ============================================================
# 🚀 SCHRITT 6: DIE KOMPLETTE PIPELINE
# ============================================================

def frag_das_dokument(frage, anzahl_chunks=3, zeige_details=True):
    """Stellt eine Frage an das Dokument und bekommt eine Antwort."""
    # 1. Frage in Embedding umwandeln
    if MODUS == "OPEN_SOURCE":
        frage_emb = embedding_modell.encode([frage])
    else:
        resp = client.embeddings.create(input=[frage], model="text-embedding-3-small")
        frage_emb = np.array([resp.data[0].embedding])

    # 2. Ähnlichste Chunks finden
    abstande, idxs = index.search(frage_emb.astype('float32'), anzahl_chunks)

    # 3. Kontext zusammenbauen
    kontext = "\n\n".join([chunks[i] for i in idxs[0]])

    # 4. Prompt bauen + Antwort generieren
    prompt = (
        "Beantworte die folgende Frage basierend NUR auf dem gegebenen Kontext.\n"
        'Wenn der Kontext die Antwort nicht enthält, sage '
        '"Das kann ich aus dem Dokument nicht beantworten."\n\n'
        f"Kontext:\n{kontext}\n\n"
        f"Frage: {frage}\n\n"
        "Antwort:"
    )

    if MODUS == "OPEN_SOURCE":
        inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
        with torch.no_grad():
            outputs = llm_modell.generate(
                **inputs, max_new_tokens=200, temperature=0.7, do_sample=True
            )
        antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=300, temperature=0.7
        )
        antwort = response.choices[0].message.content

    # 5. Ergebnis anzeigen
    print(f"❓ Frage: {frage}")
    print(f"{'─'*60}")
    print(f"🤖 Antwort: {antwort}")

    if zeige_details:
        print(f"\n{'─'*60}")
        print(f"📚 Verwendete Quellen:")
        for rang, idx in enumerate(idxs[0]):
            print(f"   {rang+1}. Chunk {idx+1}: \"{chunks[idx][:80]}...\"")
        print()

    return antwort

# --- Probiert es aus! ---
print("🎯 Eure eigene RAG-Pipeline ist bereit!\n")
frag_das_dokument("Was ist Machine Learning?")

In [ ]:
# ============================================================
# 🎮 PROBIERT VERSCHIEDENE FRAGEN!
# ============================================================
# Entkommentiert eine Zeile oder schreibt eure eigene Frage!

# frag_das_dokument("Welche Arten von maschinellem Lernen gibt es?")
# frag_das_dokument("Was sind Transformer-Modelle?")
# frag_das_dokument("Wie funktioniert OCR?")
# frag_das_dokument("Wozu braucht man Vektordatenbanken?")
# frag_das_dokument("Was hat Napoleon gemacht?")  # <-- Info NICHT im Text!

# 🔧 Eigene Frage:
# frag_das_dokument("Eure Frage hier!")

---
## 🌟 Bonus-Challenges (für Schnelle & Neugierige)

Ihr seid schon fertig? Hier sind Extra-Aufgaben:

### 🏆 Challenge 1: Parameter-Experiment
Gehe zurück zu Schritt 2 und ändere `chunk_groesse` auf 100, dann auf 1000.
Führe alles nochmal aus. Was verändert sich bei den Antworten?

### 🏆 Challenge 2: Stress-Test
Findet Fragen, bei denen die KI eine **falsche** oder **keine** Antwort gibt.
Wann funktioniert RAG gut, wann nicht?

### 🏆 Challenge 3: Prompt Engineering
Ändert den Prompt in der Funktion `frag_das_dokument()`.
Was passiert, wenn ihr schreibt: "Antworte wie ein Pirat" 🏴‍☠️?

### 🏆 Challenge 4: Eigenes Dokument
Ändert `pdf_pfad` in Schritt 1 auf ein anderes PDF auf eurem Computer
und führt das Notebook nochmal von oben aus!

In [ ]:
# ============================================================
# 🌟 BONUS: Verschiedene Chunk-Größen vergleichen
# ============================================================

print("🏆 Vergleich verschiedener Chunk-Größen:\n")
for groesse in [100, 200, 300, 500, 1000]:
    test_chunks = text_in_chunks_aufteilen(text, chunk_groesse=groesse, ueberlappung=50)
    print(f"   Chunk-Größe {groesse:>4}: {len(test_chunks):>3} Chunks | "
          f"Durchschn. Länge: {np.mean([len(c) for c in test_chunks]):.0f} Zeichen")

# ============================================================
# 🌟 BONUS: Prompt Engineering – Kreative Antworten
# ============================================================

def frag_kreativ(frage, stil="Antworte wie ein freundlicher Lehrer"):
    """Wie frag_das_dokument, aber mit anpassbarem Antwort-Stil!"""
    if MODUS == "OPEN_SOURCE":
        frage_emb = embedding_modell.encode([frage])
    else:
        resp = client.embeddings.create(input=[frage], model="text-embedding-3-small")
        frage_emb = np.array([resp.data[0].embedding])

    abstande, idxs = index.search(frage_emb.astype('float32'), 3)
    kontext = "\n\n".join([chunks[i] for i in idxs[0]])

    prompt = f"{stil}\n\nKontext: {kontext}\n\nFrage: {frage}\n\nAntwort:"

    if MODUS == "OPEN_SOURCE":
        inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
        with torch.no_grad():
            outputs = llm_modell.generate(
                **inputs, max_new_tokens=200, temperature=0.9, do_sample=True
            )
        antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    else:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": stil},
                      {"role": "user", "content": prompt}],
            max_tokens=300, temperature=0.9
        )
        antwort = response.choices[0].message.content

    print(f"🎭 Stil: [{stil}]")
    print(f"❓ {frage}")
    print(f"🤖 {antwort}\n")

# Beispiele (entkommentieren zum Ausprobieren):
# frag_kreativ("Was ist Machine Learning?", "Antworte wie ein Pirat")
# frag_kreativ("Was ist Machine Learning?", "Erkläre es einem 5-Jährigen")
# frag_kreativ("Was sind Embeddings?", "Antworte in genau 3 Sätzen")
print("💡 Entkommentiert die Zeilen oben, um kreative Antworten zu testen!")